In [ ]:
import shutil
import random
from pathlib import Path

# ==================== 경로 ====================
DATA_ROOT = Path(r"N:\개인\대원&수빈\최종 프로젝트\capture")

# ==================== 디버깅: 경로 존재 확인 ====================
print(f"DATA_ROOT 존재: {DATA_ROOT.exists()}")
print(f"images 폴더 존재: {(DATA_ROOT / 'images').exists()}")
print(f"labels 폴더 존재: {(DATA_ROOT / 'labels').exists()}")

if not (DATA_ROOT / 'images').exists():
    print("❌ images 폴더가 없습니다!")
    exit()
if not (DATA_ROOT / 'labels').exists():
    print("❌ labels 폴더가 없습니다!")
    exit()

# ==================== 분할 비율 ====================
TRAIN_RATIO = 0.7
VAL_RATIO   = 0.15
TEST_RATIO  = 0.15

# ==================== 폴더 생성 ====================
for split in ['train', 'val', 'test']:
    (DATA_ROOT / split / 'images').mkdir(parents=True, exist_ok=True)
    (DATA_ROOT / split / 'labels').mkdir(parents=True, exist_ok=True)

# ==================== 이미지 리스트 (확장자 필터링 추가) ====================
IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}

# ✅ 수정: glob('*') → 이미지 확장자만 필터링 + 파일만 선택
images = sorted([
    f for f in (DATA_ROOT / 'images').iterdir()
    if f.is_file() and f.suffix.lower() in IMG_EXTS
])

# 라벨 파일 목록 (매칭 확인용)
label_stems = {f.stem for f in (DATA_ROOT / 'labels').iterdir()
               if f.is_file() and f.suffix == '.txt'}

print(f"\n이미지 파일 수: {len(images)}")
print(f"라벨 파일 수:  {len(label_stems)}")

# ✅ 수정: 라벨이 있는 이미지만 사용
images = [img for img in images if img.stem in label_stems]
print(f"매칭된 쌍:     {len(images)}")

if len(images) == 0:
    print("❌ 매칭되는 이미지-라벨 쌍이 없습니다!")
    # 디버깅: 파일명 샘플 출력
    sample_imgs = list((DATA_ROOT / 'images').iterdir())[:5]
    sample_labs = list((DATA_ROOT / 'labels').iterdir())[:5]
    print(f"  이미지 샘플: {[f.name for f in sample_imgs]}")
    print(f"  라벨 샘플:  {[f.name for f in sample_labs]}")
    exit()

# ==================== 셔플 & 분할 ====================
random.seed(42)
random.shuffle(images)

total = len(images)
train_end = int(total * TRAIN_RATIO)
val_end   = int(total * (TRAIN_RATIO + VAL_RATIO))

splits = {
    'train': images[:train_end],
    'val':   images[train_end:val_end],
    'test':  images[val_end:]
}

print(f"\n전체: {total}장")
print(f"Train: {len(splits['train'])}장 ({TRAIN_RATIO:.0%})")
print(f"Val:   {len(splits['val'])}장 ({VAL_RATIO:.0%})")
print(f"Test:  {len(splits['test'])}장 ({TEST_RATIO:.0%})")

# ==================== 복사 ====================
def copy_split(img_list, split_name):
    success = 0
    fail = 0
    for img in img_list:
        try:
            # 이미지 복사
            dst_img = DATA_ROOT / split_name / 'images' / img.name
            shutil.copy2(str(img), str(dst_img))

            # 라벨 복사
            lbl_src = DATA_ROOT / 'labels' / (img.stem + '.txt')
            dst_lbl = DATA_ROOT / split_name / 'labels' / (img.stem + '.txt')
            shutil.copy2(str(lbl_src), str(dst_lbl))

            success += 1
        except Exception as e:
            fail += 1
            if fail <= 3:  # 처음 3개만 에러 출력
                print(f"    ⚠️ 실패: {img.name} → {e}")

    print(f"  {split_name}: {success}장 성공, {fail}장 실패")

print("\n📦 복사 시작...")
for split_name, img_list in splits.items():
    copy_split(img_list, split_name)

# ==================== 검증 ====================
print("\n📊 최종 확인:")
for split in ['train', 'val', 'test']:
    img_cnt = len(list((DATA_ROOT / split / 'images').glob('*.*')))
    lab_cnt = len(list((DATA_ROOT / split / 'labels').glob('*.txt')))
    match = "✅" if img_cnt == lab_cnt else "⚠️ 불일치!"
    print(f"  {split}: 이미지 {img_cnt}개, 라벨 {lab_cnt}개 {match}")

print("\n✅ 분할 완료!")